<a href="https://colab.research.google.com/github/ben854719/Mission-Readiness-Scoring-System-Simulation-And-Diagnostics/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import IPython

# Sample data matching the ReadinessData interface
dashboard_data = {
    "workloadPressure": 85,
    "responseTimeline": 12,
    "operationalRisk": 0.2,
    "scenarioInjects": 5,
    "readinessScore": 92,
    "bottlenecks": ["Network Latency", "Resource Allocation", "Data Sync Delay"]
}

html_content = f"""
<div style='padding: 20px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; border: 1px solid #ddd; border-radius: 8px; max-width: 500px; background-color: #f9f9f9;'>
  <h2 style='margin-top: 0; color: #333;'>Mission Readiness Dashboard</h2>

  <div style='margin-top: 16px;'>
    <strong style='color: #555;'>Readiness Score:</strong> <span style='font-size: 1.2em; font-weight: bold; color: #2e7d32;'>{dashboard_data['readinessScore']}%</span>
  </div>

  <div style='margin-top: 12px;'>
    <strong style='color: #555;'>Workload Pressure:</strong> {dashboard_data['workloadPressure']}%
  </div>

  <div style='margin-top: 12px;'>
    <strong style='color: #555;'>Response Timeline:</strong> {dashboard_data['responseTimeline']} min
  </div>

  <div style='margin-top: 12px;'>
    <strong style='color: #555;'>Operational Risk:</strong> {dashboard_data['operationalRisk']}
  </div>

  <div style='margin-top: 12px;'>
    <strong style='color: #555;'>Scenario Injects:</strong> {dashboard_data['scenarioInjects']}
  </div>

  <div style='margin-top: 20px;'>
    <strong style='color: #555;'>Bottlenecks:</strong>
    <ul style='margin-top: 8px; padding-left: 20px;'>
      {''.join([f'<li>{b}</li>' for b in dashboard_data['bottlenecks']])}
    </ul>
  </div>
</div>
"""

IPython.display.display(IPython.display.HTML(html_content))

In [9]:
!pip install --upgrade langchain-google-genai google-generativeai
!pip install --upgrade langchain-google-genai google-generativeai langgraph
!pip install langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 28.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.23
    Uninstalling langchain-core-1.2.23:
      Successfully uninstalled langchain-core-1.2.23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.1
    Uninstalling langgraph-checkpoint-4.0.1:
      Successfully uninstalled langgraph-checkpoint-4.0.1
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempti

In [10]:
import os
os.environ["LANGSMITH_API_KEY"] = "LangSmith"

In [12]:
!pip install "mcp[cli]"
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("GeminiTools")

@mcp.tool()
def search(query: str) -> list:
    # Your search logic here
    return ["Result 1", "Result 2"]

In [13]:
!cd mcp-server-demo

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [14]:
!cd mcp-server-demo && uv add langchain-google-genai langgraph langsmith

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from IPython.display import display, HTML
from typing import TypedDict, List
import os
from mcp.server.fastmcp import FastMCP
from google.colab import userdata
from langsmith import traceable
import random
import json

# Import API Key.
api_key = userdata.get("Ben856")
os.environ["GOOGLE_API_KEY"] = api_key

# Agent State Definition
class AgentState(TypedDict):
    input: str
    logs: str
    symptom: str
    diagnostic_report: str
    context: dict
    network_ok: bool
    cache_ok: bool
    token_valid: bool
    version_ok: bool
    kpi_metrics: dict
    trace_id: str

# Prompt Template rewritten for MRIDS
prompt = ChatPromptTemplate.from_template("""
    You are an agentic diagnostic system supporting a Mission Readiness
    Intelligence & Diagnostics System (MRIDS).

    Core Principles:
    - Autonomous Context Synthesis: Integrate workload pressure, response timelines,
      operational risk indicators, scenario injects, and backend simulation outputs.
    - Human‑Aligned Decision Support: Provide adaptive, non‑authoritarian insights
      that support analysts and operators without overriding human judgment.
    - Transparent Diagnostic Reasoning: Produce clear, traceable explanations grounded
      in mission‑readiness criteria and operational context.

    Signal/Symptom: {symptom}
    System Logs: {logs}

    Performance Metrics (Current vs Baseline):
    {kpi_data}

    Your task:
    1. Analyze mission‑readiness health for anomalies or degradation patterns.
    2. Evaluate KPI improvements and operational efficiency gains.
    3. Produce a diagnostic report with root causes, readiness impacts,
       and transparent reasoning aligned with modern readiness and tradecraft standards.
    """)

# Invoke Gemini Model.
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", api_key=api_key)

try:
    tools = [search]
    llm_with_tools = llm.bind_tools(tools)
except NameError:
    llm_with_tools = llm

@traceable(name="entry_node")
def entry_node(state: AgentState) -> AgentState:
    # Using the dashboard_data from the previous cell for analysis
    kpi_summary = {
        "Workload Pressure": {"current": 0.85, "baseline": 0.50},
        "Response Timeline (min)": {"current": 12, "baseline": 20},
        "Operational Risk": {"current": 0.20, "baseline": 0.35}
    }
    return {
        **state,
        "symptom": state["input"],
        "kpi_metrics": kpi_summary,
        "trace_id": f"trace-{random.randint(1000, 9999)}"
    }

@traceable(name="llm_node")
def llm_node(state: AgentState) -> AgentState:
    kpi_str = json.dumps(state["kpi_metrics"], indent=2)
    messages = prompt.format_messages(
        symptom=state["symptom"],
        logs=state.get("logs", "Normal operational telemetry"),
        kpi_data=kpi_str
    )
    response = llm_with_tools.invoke(messages)
    return {**state, "diagnostic_report": response.content}

@traceable(name="display_node")
def display_node(state: AgentState) -> AgentState:
    print("\n=== Mission Readiness Intelligence Report (Gemini Analysis) ===")
    print(f"Context Source: {state['context'].get('source', 'MRIDS Dashboard Data')}")
    print(f"\nAnalysis:\n{state['diagnostic_report']}")
    return state

workflow = StateGraph(AgentState)
workflow.add_node("entry", entry_node)
workflow.add_node("llm", llm_node)
workflow.add_node("display", display_node)
workflow.set_entry_point("entry")
workflow.add_edge("entry", "llm")
workflow.add_edge("llm", "display")
workflow.add_edge("display", END)

agent_executor = workflow.compile()

# Run the analysis on the dashboard metrics
agent_executor.invoke({
    "input": "Analyze the latest MRIDS Dashboard metrics showing 92% readiness score.",
    "logs": "Bottlenecks detected: Network Latency, Resource Allocation.",
    "context": {"source": "MRIDS Dashboard Visualization"}
})


=== Mission Readiness Intelligence Report (Gemini Analysis) ===
Context Source: MRIDS Dashboard Visualization

Analysis:
[{'type': 'text', 'text': '### **MRIDS Diagnostic Report: Mission-Readiness Analysis**\n**Report ID:** MRIDS-DX-2024-001  \n**Status:** High-Tempo Operational State  \n**System Integrity:** 92% (Nominal with Emergent Constraints)\n\n---\n\n### **1. Executive Summary**\nThe system is currently operating in an "Efficiency Paradox" state. While the overall readiness score is high (92%) and mission-critical KPIs (Response Timeline, Operational Risk) show significant improvement over baseline, the underlying infrastructure is experiencing severe strain. The high **Workload Pressure (0.85)** is beginning to trigger secondary system bottlenecks. Current operations are highly effective but potentially unsustainable without intervention.\n\n---\n\n### **2. Detailed Health & KPI Analysis**\n\n#### **A. Operational Efficiency Gains**\n*   **Response Timeline:** Improved by **4

{'input': 'Analyze the latest MRIDS Dashboard metrics showing 92% readiness score.',
 'logs': 'Bottlenecks detected: Network Latency, Resource Allocation.',
 'symptom': 'Analyze the latest MRIDS Dashboard metrics showing 92% readiness score.',
 'diagnostic_report': [{'type': 'text',
   'text': '### **MRIDS Diagnostic Report: Mission-Readiness Analysis**\n**Report ID:** MRIDS-DX-2024-001  \n**Status:** High-Tempo Operational State  \n**System Integrity:** 92% (Nominal with Emergent Constraints)\n\n---\n\n### **1. Executive Summary**\nThe system is currently operating in an "Efficiency Paradox" state. While the overall readiness score is high (92%) and mission-critical KPIs (Response Timeline, Operational Risk) show significant improvement over baseline, the underlying infrastructure is experiencing severe strain. The high **Workload Pressure (0.85)** is beginning to trigger secondary system bottlenecks. Current operations are highly effective but potentially unsustainable without interv